# 04d - Secondary Python Logistic Validation Model

---

## 1. Objective
Fit the Python logistic regression model used to generate parameter estimates for SAS-Python reproducibility validation.

---

## 2. Data Source
**Input Dataset**
- `../sas/inputs/copd_raas__cohort_copd_outcomes_extended.csv`

**Output Dataset**
- `../python/outputs/python_logistic_parameters.csv`

**Downstream Usage**
- The exported parameter file is used by `05_sas_python_validation.ipynb` for SAS/Python reproducibility validation.

The model uses the exported COPD RAAS cohort dataset from the project data pipeline and writes Python parameter estimates for validation in `05_sas_python_validation.ipynb`.

---

## 3. Study Design
This notebook uses the exported analysis cohort only to reproduce a secondary binary-outcome logistic specification for SAS-Python validation; it is not the primary clinical model.

---

## 4. Statistical Methods
A SAS-equivalent logistic regression specification is fitted in Python, and parameter estimates are exported for downstream validation.

---

## 5. Results
### 5.1 Load Cohort Dataset


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

input_path = Path("../sas/inputs/copd_raas__cohort_copd_outcomes_extended.csv")
output_path = Path("../python/outputs/python_logistic_parameters.csv")

df = pd.read_csv(input_path)
df.shape


(11964, 25)

### 5.2 Fit SAS-equivalent Logistic Regression


In [2]:
model_columns = [
    "death_event",
    "raas_pre_icu",
    "age",
    "gender",
    "sofa_score",
    "charlson_comorbidity_index",
    "chf",
    "ckd",
    "diabetes",
]

model_df = df[model_columns].dropna().copy()
model_df["gender_M"] = (model_df["gender"] == "M").astype(int)

y = model_df["death_event"].astype(float)
X = model_df[[
    "raas_pre_icu",
    "age",
    "gender_M",
    "sofa_score",
    "charlson_comorbidity_index",
    "chf",
    "ckd",
    "diabetes",
]].astype(float)
X = sm.add_constant(X, has_constant="add")

logit_result = sm.Logit(y, X).fit()
logit_result.summary()


Optimization terminated successfully.
         Current function value: 0.336311
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:            death_event   No. Observations:                11964
Model:                          Logit   Df Residuals:                    11955
Method:                           MLE   Df Model:                            8
Date:                Mon, 11 May 2026   Pseudo R-squ.:                  0.1748
Time:                        10:53:36   Log-Likelihood:                -4023.6
converged:                       True   LL-Null:                       -4875.8
Covariance Type:            nonrobust   LLR p-value:                     0.000
==============================================================================================
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                         -5.9128      0.213    -27.775      0.000      -6.330      -5.496
raas_pre_icu                  -0.3153      0.121     -2.598      0.009      -0.553      -0.077
age                            0.0221      0.003      7.845      0.000       0.017       0.028
gender_M                      -0.0911      0.058     -1.559      0.119      -0.206       0.023
sofa_score                     0.2816      0.008     33.391      0.000       0.265       0.298
charlson_comorbidity_index     0.1835      0.013     13.752      0.000       0.157       0.210
chf                           -0.0575      0.062     -0.930      0.352      -0.179       0.064
ckd                           -0.4948      0.073     -6.772      0.000      -0.638      -0.352
diabetes                      -0.3260      0.066     -4.964      0.000      -0.455      -0.197
==============================================================================================
"""

### 5.3 Export Parameter Results


The exported CSV is generated locally and is not committed to the repository. The displayed table below documents the model output used for validation.


In [3]:
params = logit_result.params
conf_int = logit_result.conf_int()

python_logistic_parameters = pd.DataFrame({
    "Variable": params.index.astype(str),
    "Python_Estimate": params.values,
    "Python_OR": np.exp(params.values),
    "Python_pvalue": logit_result.pvalues.reindex(params.index).values,
    "Python_CI_Lower": np.exp(conf_int.iloc[:, 0].reindex(params.index).values),
    "Python_CI_Upper": np.exp(conf_int.iloc[:, 1].reindex(params.index).values),
})

python_logistic_parameters["Variable"] = python_logistic_parameters["Variable"].replace({
    "const": "Intercept",
    "gender_M": "gender",
    "charlson_comorbidity_index": "charlson_comorbidity",
})

output_path.parent.mkdir(parents=True, exist_ok=True)
python_logistic_parameters.to_csv(output_path, index=False)

display(python_logistic_parameters)


,Variable,Python_Estimate,Python_OR,Python_pvalue,Python_CI_Lower,Python_CI_Upper
0,Intercept,-5.912771,0.002705,8.583911e-170,0.001782,0.004105
1,raas_pre_icu,-0.315340,0.729541,9.388912e-03,0.575062,0.925517
2,age,0.022062,1.022307,4.343568e-15,1.016688,1.027958
3,gender,-0.091077,0.912948,1.190333e-01,0.814168,1.023713
4,sofa_score,0.281622,1.325277,1.822092e-244,1.303550,1.347367
5,charlson_comorbidity,0.183531,1.201452,4.946741e-43,1.170433,1.233293
6,chf,-0.057487,0.944134,3.523441e-01,0.836412,1.065730
7,ckd,-0.494830,0.609675,1.266699e-11,0.528329,0.703545
8,diabetes,-0.326010,0.721798,6.899507e-07,0.634622,0.820949


---

## 6. Discussion
The exported Python model parameters provide an independent implementation of the logistic regression specification for comparison with SAS output in `05_sas_python_validation.ipynb`.

---

## 7. Conclusion
The notebook produces the Python parameter file required for reproducibility validation.
